# TP3 — Notebook 01 · Estaciones, cobertura y sesgo espacial

**Taller de Análisis de Datos I · Versión final propuesta · Ejecución desde `TP3/notebooks/`**

**Pregunta de investigación:** ¿Cómo cambian la disponibilidad de observaciones y el error del pronóstico meteorológico entre estaciones, estados y horizontes de 12 y 24 horas?

**Criterio central:** el análisis predictivo usa coincidencias `match_type == "interval"` y temperaturas **finitas**. La cobertura de ambas temperaturas, incluidos `fallback`, es un indicador complementario, no la muestra del modelo.

**Período descriptivo:** 10/09–01/11/2026, excluidos los días 22–26/10 por pérdida de información (recomendación docente). Los días 02–09/11 permanecen **reservados** para evaluación posterior y no se leen aquí.

**Advertencia:** este EDA describe el histórico completo, incluida la validación histórica posterior; no representa una evaluación independiente de selección de características. Los resultados de noviembre solo se analizan en los notebooks correspondientes, sin ajustar decisiones con ellos.

**Alcance:** auditoría global de metadatos, cobertura `interval` y `fallback`, MAE/RMSE/sesgo por estación y estado, comparación de horizontes en estaciones comunes y pares de fecha objetivo, evolución diaria y exportaciones para un dashboard que será la única interfaz de la presentación.

In [ ]:
# 01.1 — Bibliotecas, entorno y particiones compartidas
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_DISPONIBLE = True
except ImportError:
    CARTOPY_DISPONIBLE = False

# Cartopy es opcional. Las capas de mapas pueden requerir recursos externos.
# Si no está instalado, se representa un plano longitud/latitud sin fronteras.
USAR_CARTOPY = CARTOPY_DISPONIBLE
HORIZONTES = (12, 24)
CHUNKSIZE = 100_000

RAIZ_PROYECTO = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "dataset").is_dir() and (p / "tp3_config.py").is_file()),
    None
)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError(
        "No se encontró TP3/dataset/ y TP3/tp3_config.py. "
        "Abrí el notebook desde TP3/notebooks/."
    )
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from tp3_config import obtener_particiones, HISTORICO, TEST, EXCLUIDOS

particiones_01 = obtener_particiones(RAIZ_PROYECTO, exigir_61=True)
archivos_historicos_01 = particiones_01["historico"]
assert tuple(map(len, (particiones_01["inventario"], archivos_historicos_01,
                       particiones_01["excluidos"], particiones_01["test"]))) == (61, 48, 5, 8)
assert set(archivos_historicos_01).isdisjoint(particiones_01["test"])
assert set(archivos_historicos_01).isdisjoint(particiones_01["excluidos"])

SALIDAS_01 = RAIZ_PROYECTO / "resultados" / "dashboard" / "01_estaciones"
RUN_ID_01 = datetime.now(timezone.utc).strftime("tp3_01_%Y%m%dT%H%M%SZ")
print("Proyecto:", RAIZ_PROYECTO)
print("Inventario: 61 | histórico: 48 | excluidos: 5 | test reservado: 8")
print("Primera y última fecha histórica:", archivos_historicos_01[0].stem[-10:],
      "→", archivos_historicos_01[-1].stem[-10:])
print("Cartopy disponible:", CARTOPY_DISPONIBLE)
print("Esta ejecución NO abre archivos del test reservado.")

## 1. Protocolo de cómputo

Se procesa una sola vez cada archivo histórico, leyendo únicamente columnas necesarias por bloques. En esa pasada se obtienen simultáneamente:

- **Denominadores:** registros de cada estación y horizonte, independientemente del tipo de emparejamiento.
- **Disponibilidad:** ambas temperaturas finitas; `interval` válido; `fallback` válido.
- **Errores:** `observado − pronosticado`, su valor absoluto y su cuadrado, para muestras `interval` y `fallback` separadas.
- **Trazabilidad:** estación, estado, coordenadas, fecha de archivo, instante objetivo y cantidad de días válidos.

Los agregados usan solamente `station_id` y horizonte (más la fecha para series diarias); el maestro geográfico se calcula por separado, por modalidad de los metadatos observados, y se informa cuando una estación cambia de ubicación. **No se descartan estaciones por cobertura ni se corrigen datos sospechosos de forma automática.**

El error positivo indica que el pronóstico **subestimó** la temperatura observada; negativo indica **sobreestimación**. El MAE es siempre no negativo y menor es mejor.

In [ ]:
# 01.2 — Funciones reutilizables y validación de esquema
COLUMNAS_01 = [
    "station_id", "state", "latitude", "longitude", "horizon_target_hours",
    "match_type", "target_time_utc", "temperature_forecast_c",
    "temperature_observed_c"
]
CLAVE_EST_01 = ["station_id", "horizon_target_hours"]
CLAVE_DIA_01 = ["fecha_archivo", "station_id", "horizon_target_hours"]
CAMPOS_SUMABLES_01 = [
    "registros", "validos_temperaturas", "validos_interval", "validos_fallback",
    "validos_otro_match", "match_interval", "match_fallback", "match_unmatched",
    "n_interval", "suma_error_interval", "suma_abs_interval", "suma_sq_interval",
    "n_fallback", "suma_error_fallback", "suma_abs_fallback", "suma_sq_fallback",
    "n_todos", "suma_error_todos", "suma_abs_todos", "suma_sq_todos"
]

def porcentaje(numerador, denominador):
    denominador = pd.Series(denominador, dtype=float).replace(0, np.nan)
    return 100.0 * np.asarray(numerador, dtype=float) / denominador.to_numpy()

def derivar_metricas(tabla):
    resultado = tabla.copy()
    resultado["cobertura_temperaturas_pct"] = porcentaje(
        resultado["validos_temperaturas"], resultado["registros"])
    resultado["cobertura_interval_pct"] = porcentaje(
        resultado["validos_interval"], resultado["registros"])
    resultado["aporte_fallback_pct"] = porcentaje(
        resultado["validos_fallback"], resultado["registros"])
    for sufijo in ("interval", "fallback", "todos"):
        n = resultado[f"n_{sufijo}"].replace(0, np.nan)
        resultado[f"sesgo_{sufijo}_c"] = resultado[f"suma_error_{sufijo}"] / n
        resultado[f"MAE_{sufijo}_c"] = resultado[f"suma_abs_{sufijo}"] / n
        resultado[f"RMSE_{sufijo}_c"] = np.sqrt(
            resultado[f"suma_sq_{sufijo}"] / n)
    return resultado

def crear_mapa_base(figsize=(12, 6), valores=None, titulo=""):
    """Devuelve fig, ax, transformación; Cartopy es opcional."""
    if valores is not None and not valores.empty:
        lon = pd.to_numeric(valores["longitude"], errors="coerce").dropna()
        lat = pd.to_numeric(valores["latitude"], errors="coerce").dropna()
        extension = [lon.min() - 2, lon.max() + 2, lat.min() - 2, lat.max() + 2]             if not lon.empty and not lat.empty else [-100, -65, 23, 51]
    else:
        extension = [-100, -65, 23, 51]
    if USAR_CARTOPY:
        proyeccion = ccrs.PlateCarree()
        fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": proyeccion})
        ax.set_extent(extension, crs=proyeccion)
        try:
            for capa, estilo in [
                (cfeature.LAND, dict(facecolor="#f7f7f2")),
                (cfeature.OCEAN, dict(facecolor="#e7f2fa")),
                (cfeature.LAKES, dict(facecolor="#e7f2fa")),
                (cfeature.STATES, dict(edgecolor="#a1a1aa", linewidth=.5)),
                (cfeature.COASTLINE, dict(edgecolor="#4b5563", linewidth=.55)),
            ]:
                ax.add_feature(capa, **estilo)
        except Exception as exc:
            print("Aviso: capas Cartopy no disponibles:", exc)
        transformacion = proyeccion
    else:
        fig, ax = plt.subplots(figsize=figsize)
        ax.set_xlim(extension[:2]); ax.set_ylim(extension[2:])
        ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")
        ax.grid(alpha=.15)
        transformacion = None
    ax.set_title(titulo)
    return fig, ax, transformacion

# Validar columnas sin leer millones de registros.
for archivo in archivos_historicos_01:
    encabezado = pd.read_csv(archivo, nrows=0).columns
    faltan = set(COLUMNAS_01) - set(encabezado)
    if faltan:
        raise ValueError(f"Columnas requeridas faltantes en {archivo.name}: {sorted(faltan)}")
print(f"Esquema de columnas requeridas: OK en {len(archivos_historicos_01)} archivos históricos.")

## 2. Lectura única de los 48 CSV históricos

En cada bloque se resumen **cobertura y errores simultáneamente**. Las filas válidas `interval` también aportan, exclusivamente para el análisis pareado, su estación y tiempo objetivo. Se comprueba que sus fechas objetivo pertenezcan al archivo histórico esperado; el notebook no consulta etiquetas de noviembre.

El análisis de metadatos se realiza sobre todo el período, no únicamente sobre el primer día. La selección de una ubicación canónica por estación es descriptiva y **no elimina ni fusiona** discrepancias de origen.

In [ ]:
# 01.3 — Lectura single-pass por bloques: calidad, errores, tiempo y metadatos
partes_agrupadas_01 = []
partes_metadatos_01 = []
partes_pares_01 = []
conteos_por_archivo_01 = []

for i, archivo in enumerate(archivos_historicos_01, 1):
    fecha_archivo = pd.Timestamp(archivo.stem[-10:].replace("_", "-"))
    acumulado_archivo = {"fecha_archivo": fecha_archivo.date().isoformat(),
                         "registros_12_24": 0, "interval_validos": 0,
                         "fechas_objetivo_invalidas": 0}

    for bloque in pd.read_csv(
        archivo, usecols=COLUMNAS_01, chunksize=CHUNKSIZE, low_memory=False
    ):
        bloque = bloque.loc[bloque["horizon_target_hours"].isin(HORIZONTES)].copy()
        if bloque.empty:
            continue
        acumulado_archivo["registros_12_24"] += len(bloque)
        if bloque["station_id"].isna().any():
            raise ValueError(f"ID de estación faltante: {archivo.name}")

        pron = pd.to_numeric(bloque["temperature_forecast_c"], errors="coerce")
        obs = pd.to_numeric(bloque["temperature_observed_c"], errors="coerce")
        ambos = np.isfinite(pron) & np.isfinite(obs)
        interval = bloque["match_type"].eq("interval")
        fallback = bloque["match_type"].eq("fallback")
        valido_interval = ambos & interval
        valido_fallback = ambos & fallback
        valido_otro = ambos & ~(interval | fallback)
        error = (obs - pron).where(ambos, 0.0)
        abs_error = error.abs()
        sq_error = error * error

        bloque["registros"] = 1
        bloque["validos_temperaturas"] = ambos.astype("int8")
        bloque["validos_interval"] = valido_interval.astype("int8")
        bloque["validos_fallback"] = valido_fallback.astype("int8")
        bloque["validos_otro_match"] = valido_otro.astype("int8")
        for nombre, indicador in [
            ("match_interval", interval), ("match_fallback", fallback),
            ("match_unmatched", bloque["match_type"].eq("unmatched"))
        ]:
            bloque[nombre] = indicador.astype("int8")
        for sufijo, selector in [
            ("interval", valido_interval),
            ("fallback", valido_fallback),
            ("todos", ambos)
        ]:
            bloque[f"n_{sufijo}"] = selector.astype("int8")
            bloque[f"suma_error_{sufijo}"] = error.where(selector, 0.0)
            bloque[f"suma_abs_{sufijo}"] = abs_error.where(selector, 0.0)
            bloque[f"suma_sq_{sufijo}"] = sq_error.where(selector, 0.0)

        bloque["fecha_archivo"] = fecha_archivo.date().isoformat()
        parcial = (bloque.groupby(CLAVE_DIA_01, dropna=False, observed=True)
                   [CAMPOS_SUMABLES_01].sum().reset_index())
        partes_agrupadas_01.append(parcial)

        # Metadatos: conteos por combinación, sin usarlos como clave de métricas.
        geos = bloque[["station_id", "state", "latitude", "longitude"]].copy()
        geos["latitude"] = pd.to_numeric(geos["latitude"], errors="coerce").round(5)
        geos["longitude"] = pd.to_numeric(geos["longitude"], errors="coerce").round(5)
        geos = (geos.groupby(["station_id", "state", "latitude", "longitude"],
                              dropna=False, observed=True).size()
                .rename("frecuencia").reset_index())
        partes_metadatos_01.append(geos)

        # Pareado por misma estación y tiempo objetivo: estudio descriptivo.
        if valido_interval.any():
            fechas = pd.to_datetime(
                bloque.loc[valido_interval, "target_time_utc"],
                utc=True, errors="coerce"
            )
            if fechas.isna().any():
                acumulado_archivo["fechas_objetivo_invalidas"] += int(fechas.isna().sum())
                raise ValueError(f"Target UTC inválido en registros interval: {archivo.name}")
            if not (fechas.dt.date == fecha_archivo.date()).all():
                raise ValueError(
                    f"Fecha objetivo distinta de fecha de archivo: {archivo.name}. "
                    "Revisar la cohorte sin incorporarla automáticamente."
                )
            sub = pd.DataFrame({
                "station_id": bloque.loc[valido_interval, "station_id"].to_numpy(),
                "target_time_utc": fechas.to_numpy(),
                "horizon_target_hours": bloque.loc[valido_interval,
                                                       "horizon_target_hours"].to_numpy(),
                "error_abs_c": abs_error.loc[valido_interval].to_numpy()
            })
            partes_pares_01.append(sub)
            acumulado_archivo["interval_validos"] += len(sub)

    conteos_por_archivo_01.append(acumulado_archivo)
    if i % 8 == 0 or i == len(archivos_historicos_01):
        print(f"Procesados {i}/{len(archivos_historicos_01)} archivos históricos")

assert partes_agrupadas_01 and partes_metadatos_01 and partes_pares_01
print(f"Lectura de {len(archivos_historicos_01)} CSV históricos completa, sin abrir el test.")

In [ ]:
# 01.4 — Agregación temporal, geográfica y estadística
por_bloque = pd.concat(partes_agrupadas_01, ignore_index=True)
df_estacion_dia_01 = (por_bloque.groupby(CLAVE_DIA_01, as_index=False, observed=True)
                      [CAMPOS_SUMABLES_01].sum())
df_estacion_dia_01 = derivar_metricas(df_estacion_dia_01)
df_estacion_dia_01["dia_con_interval"] = (df_estacion_dia_01["n_interval"] > 0).astype(int)

df_metricas_estacion_01 = (
    df_estacion_dia_01.groupby(CLAVE_EST_01, as_index=False, observed=True)
    [CAMPOS_SUMABLES_01].sum()
)
dias_01 = (df_estacion_dia_01.groupby(CLAVE_EST_01, as_index=False)
           .agg(dias_con_registros=("fecha_archivo", "nunique"),
                dias_con_interval=("dia_con_interval", "sum")))
df_metricas_estacion_01 = derivar_metricas(
    df_metricas_estacion_01.merge(dias_01, on=CLAVE_EST_01, validate="one_to_one")
)

# Maestro canónico: ubicación más frecuente (redondeo a 5 decimales),
# con conteo explícito de variantes y ausencia de georreferenciación.
metadatos_01 = (pd.concat(partes_metadatos_01, ignore_index=True)
    .groupby(["station_id", "state", "latitude", "longitude"],
             dropna=False, observed=True, as_index=False)["frecuencia"].sum())
canonicos = (metadatos_01.sort_values(
    ["station_id", "frecuencia", "state"], ascending=[True, False, True],
    na_position="last")
    .drop_duplicates("station_id")
    [["station_id", "state", "latitude", "longitude"]].copy())
variants = metadatos_01.groupby("station_id").agg(
    variantes_metadatos=("frecuencia", "size"),
    variantes_estado=("state", "nunique"),
    variantes_latitud=("latitude", "nunique"),
    variantes_longitud=("longitude", "nunique"),
    registros_metadatos=("frecuencia", "sum")
).reset_index()
variants["metadatos_inconsistentes"] = (
    (variants["variantes_metadatos"] > 1)
).astype(bool)
df_dim_estaciones_01 = canonicos.merge(variants, on="station_id", validate="one_to_one")
df_dim_estaciones_01["coordenadas_disponibles"] = (
    df_dim_estaciones_01[["latitude", "longitude"]].notna().all(axis=1)
)

# La geografía se adjunta SOLO después de agrupar por estación/horizonte.
df_metricas_estacion_01 = df_metricas_estacion_01.merge(
    df_dim_estaciones_01[["station_id", "state", "latitude", "longitude",
                         "metadatos_inconsistentes", "variantes_metadatos"]],
    on="station_id", how="left", validate="many_to_one"
)

df_serie_diaria_01 = derivar_metricas(
    df_estacion_dia_01.groupby(["fecha_archivo", "horizon_target_hours"],
                                as_index=False)[CAMPOS_SUMABLES_01].sum()
)
df_global_horizonte_01 = derivar_metricas(
    df_metricas_estacion_01.groupby("horizon_target_hours", as_index=False)
    [CAMPOS_SUMABLES_01].sum()
)
df_estado_horizonte_01 = derivar_metricas(
    df_metricas_estacion_01.groupby(["state", "horizon_target_hours"],
                                     dropna=False, as_index=False)[CAMPOS_SUMABLES_01].sum()
)
n_est_estado = (df_metricas_estacion_01.groupby(["state", "horizon_target_hours"],
                  dropna=False)["station_id"].nunique().rename("estaciones").reset_index())
df_estado_horizonte_01 = df_estado_horizonte_01.merge(
    n_est_estado, on=["state", "horizon_target_hours"], validate="one_to_one"
)

# Una estación con muchas filas no debe confundirse con una muestra independiente.
df_macro_horizonte_01 = (
    df_metricas_estacion_01.loc[df_metricas_estacion_01["n_interval"] > 0]
    .groupby("horizon_target_hours")
    .agg(estaciones_con_interval=("station_id", "nunique"),
         macro_MAE_interval_c=("MAE_interval_c", "mean"),
         macro_sesgo_interval_c=("sesgo_interval_c", "mean"),
         mediana_MAE_estaciones_c=("MAE_interval_c", "median"))
    .reset_index()
)

# Controles de reconciliación; no se exige que un modelo mejore.
assert df_metricas_estacion_01[CLAVE_EST_01].duplicated().sum() == 0
assert df_dim_estaciones_01["station_id"].is_unique
assert df_estacion_dia_01[CAMPOS_SUMABLES_01].sum()["n_interval"] == 854009, (
    "La cantidad histórica interval difiere de 854009: investigar contra Notebook 00."
)
assert df_global_horizonte_01.set_index("horizon_target_hours")["n_interval"].to_dict() == {
    12: 427009, 24: 427000
}
assert (df_metricas_estacion_01["validos_interval"]
        <= df_metricas_estacion_01["validos_temperaturas"]).all()
assert (df_metricas_estacion_01["validos_temperaturas"]
        <= df_metricas_estacion_01["registros"]).all()
assert (df_metricas_estacion_01["validos_fallback"]
        + df_metricas_estacion_01["validos_interval"]
        + df_metricas_estacion_01["validos_otro_match"]
        == df_metricas_estacion_01["validos_temperaturas"]).all()
assert len(df_serie_diaria_01) == 48 * 2

SALIDAS_01.mkdir(parents=True, exist_ok=True)

print("ESTACIONES DEL HISTÓRICO:", len(df_dim_estaciones_01))
print("Estaciones con metadatos variables:",
      int(df_dim_estaciones_01["metadatos_inconsistentes"].sum()))
print("Registros históricos interval:", int(df_global_horizonte_01["n_interval"].sum()))
print("MÉTRICAS POR HORIZONTE (micro, ponderadas por fila):")
display(df_global_horizonte_01[[
    "horizon_target_hours", "registros", "validos_temperaturas",
    "validos_interval", "validos_fallback", "cobertura_temperaturas_pct",
    "cobertura_interval_pct", "MAE_interval_c", "RMSE_interval_c", "sesgo_interval_c"
]].round(3))
print("MÉTRICAS MACRO (cada estación cuenta una vez):")
display(df_macro_horizonte_01.round(3))

## 3. Disponibilidad: `interval` frente a todas las temperaturas disponibles

La **cobertura principal** es `interval_validos / registros` para horizontes de 12 y 24 h. La cobertura total de ambas temperaturas puede ser mayor porque incluye emparejamientos `fallback`; esa diferencia se presenta explícitamente. Las proporciones por estación tienen denominadores distintos: una estación con poca cobertura puede tener una mediana o un MAE extremos sin que ello implique mayor representatividad.

Los umbrales del gráfico son **descriptivos**, no reglas de exclusión de estaciones del modelo.

In [ ]:
# 01.5 — Resumen de cobertura, interval vs fallback e histogramas comparables
res_cob_01 = (df_metricas_estacion_01.groupby("horizon_target_hours")
    [["cobertura_interval_pct", "cobertura_temperaturas_pct", "aporte_fallback_pct"]]
    .agg(["count", "min", "median", "max"]))
print("COBERTURA POR ESTACIÓN — DISTRIBUCIÓN DESCRIPTIVA:")
display(res_cob_01.round(2))

print("ESTACIONES DE MENOR COBERTURA INTERVAL (12 h):")
display(df_metricas_estacion_01.loc[
    df_metricas_estacion_01["horizon_target_hours"].eq(12)
].nsmallest(12, "cobertura_interval_pct")[[
    "station_id", "state", "registros", "validos_interval", "validos_fallback",
    "cobertura_interval_pct", "cobertura_temperaturas_pct", "dias_con_interval"
]].round(2))

fig, ax = plt.subplots(figsize=(11, 5))
bordes_cobertura_01 = np.arange(0, 110, 10)
for h, color in [(12, "#1f77b4"), (24, "#e07a24")]:
    valores = df_metricas_estacion_01.loc[
        df_metricas_estacion_01["horizon_target_hours"].eq(h),
        "cobertura_interval_pct"
    ].dropna()
    ax.hist(valores, bins=bordes_cobertura_01, alpha=.5, color=color,
            label=f"{h} h · {len(valores)} estaciones", edgecolor="white")
ax.set(xlabel="Cobertura interval válida sobre registros (%)",
       ylabel="Número de estaciones",
       title="Distribución de cobertura interval por estación — histórico de 48 días")
ax.set_xticks(np.arange(0, 101, 10)); ax.set_xlim(0, 100)
ax.grid(axis="y", alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(SALIDAS_01 / "12_histograma_cobertura_interval.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

## 4. Distribución geográfica

Una cantidad similar de estaciones por estado no equivale a cobertura uniforme ni a un muestreo aleatorio del territorio. Los mapas muestran los puntos disponibles, con una ubicación canónica por estación. Si las coordenadas cambian, la cantidad de variantes queda registrada en el maestro de estaciones.

El mapa de cobertura usa **solo** `interval` válido. Si Cartopy no está instalado, se utiliza un gráfico de coordenadas sin fronteras para no bloquear la ejecución del notebook.

In [ ]:
# 01.6 — Distribución por estado, ubicación y mapa principal de cobertura
por_estado_01 = (df_dim_estaciones_01.groupby("state", dropna=False)
                 ["station_id"].nunique().sort_values(ascending=False))
print("ESTACIONES POR ESTADO/REGIÓN:")
display(por_estado_01.to_frame("estaciones"))

fig, ax = plt.subplots(figsize=(12, 4.5))
por_estado_01.plot(kind="bar", ax=ax, color="#3776ab")
ax.set(xlabel="Estado/región", ylabel="Estaciones únicas",
       title="Distribución geográfica de las estaciones históricas")
ax.grid(axis="y", alpha=.25)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(SALIDAS_01 / "13_distribucion_estaciones_estado.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

ubicaciones_01 = df_dim_estaciones_01.dropna(subset=["latitude", "longitude"])
fig, ax, transformacion = crear_mapa_base(
    valores=ubicaciones_01,
    titulo="Distribución de estaciones históricas (ubicación canónica)"
)
argumentos = {"transform": transformacion} if transformacion is not None else {}
ax.scatter(ubicaciones_01["longitude"], ubicaciones_01["latitude"],
           s=18, alpha=.8, c="#245a81", edgecolor="white", linewidth=.25,
           zorder=4, **argumentos)
fig.tight_layout()
fig.savefig(SALIDAS_01 / "14_mapa_posicion_estaciones.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

mapa_cobertura_01 = df_metricas_estacion_01.loc[
    df_metricas_estacion_01["horizon_target_hours"].eq(12)
].dropna(subset=["latitude", "longitude", "cobertura_interval_pct"])
fig, ax, transformacion = crear_mapa_base(
    valores=mapa_cobertura_01,
    titulo="Cobertura interval por estación — 12 h · histórico sin días excluidos"
)
argumentos = {"transform": transformacion} if transformacion is not None else {}
pts = ax.scatter(mapa_cobertura_01["longitude"], mapa_cobertura_01["latitude"],
                 c=mapa_cobertura_01["cobertura_interval_pct"],
                 cmap="viridis", vmin=0, vmax=100, s=30, alpha=.86,
                 edgecolor="white", linewidth=.15, **argumentos)
fig.colorbar(pts, ax=ax, label="Cobertura interval válida (%)", shrink=.8)
fig.tight_layout()
fig.savefig(SALIDAS_01 / "15_mapa_cobertura_estaciones.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

## 5. ¿Qué precisión tiene el pronóstico original por estación?

Se calculan tres métricas con la convención `error = observado − pronosticado`:

- **Sesgo (°C):** promedio firmado. Positivo indica subestimación; negativo, sobreestimación.
- **MAE (°C):** promedio de errores absolutos.
- **RMSE (°C):** penaliza errores grandes.

Se presentan métricas globales **micro** (cada pronóstico aporta una fila) y resúmenes **macro** (cada estación con registros válidos cuenta una vez). No deben interpretarse como estimaciones equivalentes ni como evidencia de independencia de estaciones, horas y días.

In [ ]:
# 01.7 — Distribución de errores y selección descriptiva de estaciones críticas
print("MÉTRICAS DE PRECISIÓN GLOBAL POR HORIZONTE:")
display(df_global_horizonte_01[[
    "horizon_target_hours", "n_interval", "MAE_interval_c",
    "RMSE_interval_c", "sesgo_interval_c"
]].round(4))

# Umbrales descriptivos: 100 observaciones Y 7 fechas distintas.
# Se conserva la muestra original; solo cambian los rankings gráficos.
UMBRAL_REGISTROS_01 = 100
UMBRAL_DIAS_01 = 7
elegibles_01 = df_metricas_estacion_01.loc[
    (df_metricas_estacion_01["n_interval"] >= UMBRAL_REGISTROS_01)
    & (df_metricas_estacion_01["dias_con_interval"] >= UMBRAL_DIAS_01)
].copy()
print("Estaciones/horizontes con soporte descriptivo:", len(elegibles_01),
      "de", len(df_metricas_estacion_01))
print("MAYOR MAE (no implica falla del sensor ni fenómeno causal):")
display(elegibles_01.nlargest(10, "MAE_interval_c")[[
    "station_id", "state", "horizon_target_hours", "n_interval",
    "dias_con_interval", "MAE_interval_c", "sesgo_interval_c"
]].round(3))
print("MAYOR SESGO ABSOLUTO (signo conservado):")
elegibles_01["sesgo_abs_c"] = elegibles_01["sesgo_interval_c"].abs()
display(elegibles_01.nlargest(10, "sesgo_abs_c")[[
    "station_id", "state", "horizon_target_hours", "n_interval",
    "dias_con_interval", "MAE_interval_c", "sesgo_interval_c"
]].round(3))

valores_sesgo = [
    elegibles_01.loc[elegibles_01["horizon_target_hours"].eq(h), "sesgo_interval_c"].dropna()
    for h in HORIZONTES
]
maxabs = max((float(v.abs().max()) for v in valores_sesgo if not v.empty), default=1.0)
maxabs = max(0.5, np.ceil(maxabs / .5) * .5)
bordes_sesgo_01 = np.linspace(-maxabs, maxabs, 33)
fig, ax = plt.subplots(figsize=(11, 5))
for v, h, color in zip(valores_sesgo, HORIZONTES, ["#1f77b4", "#e07a24"]):
    ax.hist(v, bins=bordes_sesgo_01, alpha=.50, color=color, label=f"{h} h · {len(v)} estaciones")
ax.axvline(0, color="black", lw=1, ls="--")
ax.set(xlabel="Sesgo medio por estación (°C)", ylabel="Estaciones",
       title="Distribución del sesgo — estaciones con soporte suficiente")
ax.grid(axis="y", alpha=.20); ax.legend(); fig.tight_layout()
fig.savefig(SALIDAS_01 / "16_histograma_sesgos_estaciones.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

## 6. Comparación entre horizontes con estaciones comunes y objetivos pareados

Comparar dos medias globales puede confundir diferencias de cobertura con diferencias de desempeño. Por eso se conservan **dos análisis complementarios**:

1. **Estaciones comunes:** MAE de 12 h frente a 24 h en las mismas estaciones que cumplen un umbral descriptivo de registros y días.
2. **Pares estación–tiempo objetivo:** cuando ambos horizontes están disponibles para la **misma estación y mismo instante objetivo**, comparamos errores absolutos. Si existen varias emisiones para un mismo objetivo y horizonte, se promedian sus errores absolutos antes del pareado; se informa cuántas claves se vieron afectadas.

Un pareado por objetivo no equivale a comparar emisiones simultáneas: los horizontes corresponden a pronósticos hechos con distinta anticipación. Los grupos tampoco son observaciones estadísticamente independientes.

In [ ]:
# 01.8 — Estaciones comunes y pareado por target_time_utc
filtradas_01 = elegibles_01.pivot(
    index="station_id", columns="horizon_target_hours", values="MAE_interval_c"
).dropna(subset=list(HORIZONTES))
filtradas_01.columns = [f"MAE_{h}h_c" for h in filtradas_01.columns]
filtradas_01 = filtradas_01.reset_index()
filtradas_01["diferencia_MAE_24_menos_12_c"] = (
    filtradas_01["MAE_24h_c"] - filtradas_01["MAE_12h_c"]
)
print("ESTACIONES COMUNES CON SOPORTE DESCRIPTIVO:", len(filtradas_01))
display(filtradas_01[["MAE_12h_c", "MAE_24h_c",
                           "diferencia_MAE_24_menos_12_c"]].describe().round(3))

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(filtradas_01["MAE_12h_c"], filtradas_01["MAE_24h_c"],
           s=18, alpha=.5, c="#2563a6")
limite = max(float(filtradas_01[["MAE_12h_c", "MAE_24h_c"]].max().max()) * 1.04, 1)
ax.plot([0, limite], [0, limite], linestyle="--", color="#7d7d7d", label="MAE iguales")
ax.set(xlim=(0,limite), ylim=(0,limite), xlabel="MAE 12 h (°C)",
       ylabel="MAE 24 h (°C)", title="MAE por estación: mismas estaciones, dos horizontes")
ax.legend(); ax.grid(alpha=.15); fig.tight_layout()
fig.savefig(SALIDAS_01 / "17_comparacion_mae_12_vs_24.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

raw_pares_01 = pd.concat(partes_pares_01, ignore_index=True)
claves_par_01 = ["station_id", "target_time_utc", "horizon_target_hours"]
multiplos_01 = int(raw_pares_01.duplicated(claves_par_01, keep=False).sum())
pares_unicos_01 = (raw_pares_01.groupby(claves_par_01, as_index=False)
                   .agg(error_abs_c=("error_abs_c", "mean"),
                        emisiones=("error_abs_c", "size")))
pivot_pares_01 = pares_unicos_01.pivot(
    index=["station_id", "target_time_utc"],
    columns="horizon_target_hours", values="error_abs_c"
)
pares_01 = pivot_pares_01.dropna(subset=list(HORIZONTES)).reset_index()
pares_01 = pares_01.rename(columns={12: "abs_12h_c", 24: "abs_24h_c"})
pares_01["diferencia_abs_24_menos_12_c"] = (
    pares_01["abs_24h_c"] - pares_01["abs_12h_c"]
)
pares_01["fecha_objetivo"] = pd.to_datetime(pares_01["target_time_utc"], utc=True).dt.date.astype(str)
print("Registros afectados por claves estación+objetivo+horizonte repetidas:", multiplos_01)
print("PARES CON AMBOS HORIZONTES:", len(pares_01))
if not pares_01.empty:
    print("Diferencia media |error 24 h| − |error 12 h| en pares:",
          round(pares_01["diferencia_abs_24_menos_12_c"].mean(), 4), "°C")

df_pares_estacion_01 = (pares_01.groupby("station_id", as_index=False)
                       .agg(n_objetivos_pareados=("target_time_utc", "size"),
                            media_abs_12h_c=("abs_12h_c", "mean"),
                            media_abs_24h_c=("abs_24h_c", "mean"),
                            diferencia_24_menos_12_c=("diferencia_abs_24_menos_12_c", "mean")))
df_pares_diario_01 = (pares_01.groupby("fecha_objetivo", as_index=False)
                      .agg(n_objetivos_pareados=("station_id", "size"),
                           media_abs_12h_c=("abs_12h_c", "mean"),
                           media_abs_24h_c=("abs_24h_c", "mean"),
                           diferencia_24_menos_12_c=("diferencia_abs_24_menos_12_c", "mean")))

## 7. Cobertura y error en el tiempo

Se muestran los 48 días históricos sin interpolar los cinco días excluidos. Una línea que cruza del 21 al 27 de octubre sugeriría continuidad donde no existe observación válida; por eso se insertan días faltantes como `NaN` y se señalan con una franja.

La variabilidad temporal refleja cambios de condiciones, cobertura y mezcla de estaciones, no necesariamente una degradación del proveedor o una tendencia climática.

In [ ]:
# 01.9 — Serie diaria con discontinuidad real de 22–26/10
serie_diaria_graf_01 = df_serie_diaria_01.copy()
serie_diaria_graf_01["fecha_archivo"] = pd.to_datetime(serie_diaria_graf_01["fecha_archivo"])
fechas_calendario_01 = pd.date_range("2026-09-10", "2026-11-01", freq="D")

fig, ejes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for h, color in [(12, "#1f77b4"), (24, "#e07a24")]:
    d = serie_diaria_graf_01.loc[serie_diaria_graf_01["horizon_target_hours"].eq(h)]
    d = d.set_index("fecha_archivo").reindex(fechas_calendario_01)
    ejes[0].plot(d.index, d["cobertura_interval_pct"], color=color,
                 label=f"{h} h", marker=".", markersize=4)
    ejes[1].plot(d.index, d["MAE_interval_c"], color=color,
                 label=f"{h} h", marker=".", markersize=4)
for ax in ejes:
    ax.axvspan(pd.Timestamp("2026-10-22"), pd.Timestamp("2026-10-27"),
               color="#c2410c", alpha=.12, label="22–26/10 excluidos")
    ax.grid(alpha=.20); ax.legend(loc="best", ncol=3)
ejes[0].set(ylabel="Cobertura interval válida (%)",
            title="Cobertura diaria histórica: dos horizontes")
ejes[0].set_ylim(0, 100)
ejes[1].set(xlabel="Fecha de archivo (histórico)", ylabel="MAE original interval (°C)",
            title="Error absoluto medio diario: sin interpolar los días excluidos")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(SALIDAS_01 / "18_serie_cobertura_y_mae.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

## 8. Mapa del sesgo y diferencias por estado

El sesgo cercano a cero puede esconder grandes errores absolutos o compensación de signos. Por eso el mapa de sesgo se acompaña de MAE, soporte de registros y fechas. La escala de colores es **divergente y centrada en cero**; el tono representa el signo del sesgo, no su significación estadística.

Los rankings estatales son descriptivos y dependen de las estaciones que aparecen en la muestra. No deben leerse como comparaciones causales entre estados.

In [ ]:
# 01.10 — Estado/región, puntos críticos y mapa de sesgo a 12 h
print("ESTADOS — MAE Y SESGO ORIGINAL INTERVAL:")
display(df_estado_horizonte_01[[
    "state", "horizon_target_hours", "estaciones", "n_interval",
    "cobertura_interval_pct", "MAE_interval_c", "sesgo_interval_c"
]].sort_values(["horizon_target_hours", "MAE_interval_c"],
                ascending=[True, False]).head(16).round(3))

mapa_sesgo_01 = elegibles_01.loc[
    elegibles_01["horizon_target_hours"].eq(12)
].dropna(subset=["latitude", "longitude", "sesgo_interval_c"])
limite_color_01 = max(float(mapa_sesgo_01["sesgo_interval_c"].abs().max())
                      if not mapa_sesgo_01.empty else 0, .25)
fig, ax, transformacion = crear_mapa_base(
    valores=mapa_sesgo_01,
    titulo="Sesgo original por estación — 12 h (100+ registros, 7+ días)"
)
argumentos = {"transform": transformacion} if transformacion is not None else {}
pts = ax.scatter(mapa_sesgo_01["longitude"], mapa_sesgo_01["latitude"],
                 c=mapa_sesgo_01["sesgo_interval_c"], cmap="RdBu_r",
                 norm=TwoSlopeNorm(vcenter=0, vmin=-limite_color_01, vmax=limite_color_01),
                 s=33, edgecolor="white", linewidth=.2, alpha=.92, **argumentos)
fig.colorbar(pts, ax=ax, shrink=.82, label="Sesgo (observado − pronosticado), °C")
fig.tight_layout()
fig.savefig(SALIDAS_01 / "19_mapa_sesgo_estaciones.png", dpi=175, bbox_inches="tight")
plt.show(); plt.close(fig)

## 9. Control de coherencia y exportación para el dashboard

Se generan **datos reutilizables**, no solo PNG. Las tablas son consumibles por el futuro dashboard interactivo, sin volver a leer los archivos originales durante la exposición.

**Carpeta:** `TP3/resultados/dashboard/01_estaciones/` (coherente con `TP3/resultados/dashboard/00_calidad_datos/`).

**Contrato de salida:**

- `01_dim_estaciones.csv`: una fila por estación; ubicación canónica y alertas de metadatos variables.
- `02_fact_metricas_estacion_horizonte.csv`: cobertura, días disponibles, sesgo, MAE y RMSE por estación y horizonte.
- `03_fact_serie_temporal_diaria.csv`: conteos y métricas por día y horizonte.
- `04_fact_metricas_estado_horizonte.csv`: comparaciones regionales.
- `05_fact_cobertura_estacion_diaria.csv`: cobertura dinámica por fecha y estación.
- `06_fact_horizontes_pareados_estacion.csv` y `07_fact_horizontes_pareados_dia.csv`: comparación 12/24 sobre objetivos comunes.
- `08_resumen_global_horizonte.csv` y `09_resumen_macro_horizonte.csv`: micro y macro-MAE.
- `10_estaciones_criticas.csv`: selección descriptiva de estaciones con baja cobertura o error elevado y soporte indicado.
- `11_resumen_estaciones.json`: indicadores, criterios, advertencias y trazabilidad.
- `12_*` a `19_*`: ocho PNG de apoyo, generados una sola vez junto a sus gráficos.

**Sin datos del test:** ninguna exportación de este notebook contiene predicciones, observaciones ni métricas de noviembre.

In [ ]:
# 01.11 — Reconciliación opcional contra Notebook 00 y exportaciones
ruta_00 = (RAIZ_PROYECTO / "resultados" / "dashboard"
           / "00_calidad_datos" / "04_auditoria_estaciones.csv")
if ruta_00.is_file():
    ref_00 = pd.read_csv(ruta_00)
    ref_01 = (df_metricas_estacion_01.groupby("station_id", as_index=False)
              [["registros", "validos_interval"]].sum())
    union_00_01 = ref_00[["station_id", "registros", "validos_interval"]].merge(
        ref_01, on="station_id", how="outer", suffixes=("_00", "_01"),
        indicator=True, validate="one_to_one")
    assert union_00_01["_merge"].eq("both").all(), (
        "Estaciones distintas entre 00 y 01: verificar versiones de los archivos.")
    for variable in ("registros", "validos_interval"):
        assert (union_00_01[f"{variable}_00"]
                == union_00_01[f"{variable}_01"]).all(), (
            f"No reconcilian {variable} entre 00 y 01")
    print("Reconciliación contra Notebook 00: CORRECTA, estación por estación")
else:
    print("Notebook 00 aún no exportado: comparación cruzada pendiente, no inventada.")

# Identificar filas descriptivamente críticas sin eliminar estaciones del modelado.
df_criticas_01 = df_metricas_estacion_01.copy()
df_criticas_01["baja_cobertura_50pct"] = (
    df_criticas_01["cobertura_interval_pct"] < 50)
# El percentil se define solo entre estaciones con soporte interpretativo.
p90_mae_01 = (elegibles_01.groupby("horizon_target_hours")
              ["MAE_interval_c"].quantile(.90).to_dict())
df_criticas_01["MAE_umbral_p90_c"] = (
    df_criticas_01["horizon_target_hours"].map(p90_mae_01))
df_criticas_01["MAE_alto_soporte"] = (
    (df_criticas_01["n_interval"] >= UMBRAL_REGISTROS_01)
    & (df_criticas_01["dias_con_interval"] >= UMBRAL_DIAS_01)
    & (df_criticas_01["MAE_interval_c"] >= df_criticas_01["MAE_umbral_p90_c"])
)

tablas_exportar_01 = {
    "01_dim_estaciones.csv": df_dim_estaciones_01,
    "02_fact_metricas_estacion_horizonte.csv": df_metricas_estacion_01,
    "03_fact_serie_temporal_diaria.csv": df_serie_diaria_01,
    "04_fact_metricas_estado_horizonte.csv": df_estado_horizonte_01,
    "05_fact_cobertura_estacion_diaria.csv": df_estacion_dia_01,
    "06_fact_horizontes_pareados_estacion.csv": df_pares_estacion_01,
    "07_fact_horizontes_pareados_dia.csv": df_pares_diario_01,
    "08_resumen_global_horizonte.csv": df_global_horizonte_01,
    "09_resumen_macro_horizonte.csv": df_macro_horizonte_01,
    "10_estaciones_criticas.csv": df_criticas_01,
}
SALIDAS_01.mkdir(parents=True, exist_ok=True)
for archivo, tabla in tablas_exportar_01.items():
    tabla.to_csv(SALIDAS_01 / archivo, index=False, encoding="utf-8-sig")

resumen_json_01 = {
    "run_id": RUN_ID_01,
    "historico_archivos": 48,
    "excluidos_archivos": 5,
    "reservados_test_archivos": 8,
    "primera_fecha_historica": "2026-09-10",
    "ultima_fecha_historica": "2026-11-01",
    "excluidos": ["2026-10-%02d" % d for d in range(22, 27)],
    "estaciones_historicas": int(df_dim_estaciones_01["station_id"].nunique()),
    "estaciones_metadatos_inconsistentes": int(df_dim_estaciones_01["metadatos_inconsistentes"].sum()),
    "estaciones_sin_coordenadas_canonicas": int((~df_dim_estaciones_01["coordenadas_disponibles"]).sum()),
    "registros_interval_historicos": int(df_global_horizonte_01["n_interval"].sum()),
    "registros_interval_12h": int(df_global_horizonte_01.loc[
        df_global_horizonte_01["horizon_target_hours"].eq(12), "n_interval"].iloc[0]),
    "registros_interval_24h": int(df_global_horizonte_01.loc[
        df_global_horizonte_01["horizon_target_hours"].eq(24), "n_interval"].iloc[0]),
    "pares_station_target": int(len(pares_01)),
    "registros_con_clave_pareado_repetida": multiplos_01,
    "criterio": "histórico por archivo 48 días; solo 12/24 horas; principal interval y dos temperaturas finitas",
    "cobertura": "validos_interval/registros de estación/horizonte; fallback separado",
    "sesgo": "observado − pronosticado",
    "advertencia": "EDA descriptivo histórico; no se usa test reservado ni se infiere causalidad",
    "umbral_ranking": f">={UMBRAL_REGISTROS_01} pronósticos y >={UMBRAL_DIAS_01} días con interval",
}
with open(SALIDAS_01 / "11_resumen_estaciones.json", "w", encoding="utf-8") as f:
    json.dump(resumen_json_01, f, indent=2, ensure_ascii=False)

print("ARCHIVOS DE DATOS EXPORTADOS:", SALIDAS_01)
for nombre in tablas_exportar_01:
    assert (SALIDAS_01 / nombre).is_file()
assert (SALIDAS_01 / "11_resumen_estaciones.json").is_file()
print("Tablas: 10 CSV | Indicadores: 1 JSON")

In [ ]:
# 01.12 — Comprobación final de visualizaciones y entregables
nombres_png_01 = [
    "12_histograma_cobertura_interval.png",
    "13_distribucion_estaciones_estado.png",
    "14_mapa_posicion_estaciones.png",
    "15_mapa_cobertura_estaciones.png",
    "16_histograma_sesgos_estaciones.png",
    "17_comparacion_mae_12_vs_24.png",
    "18_serie_cobertura_y_mae.png",
    "19_mapa_sesgo_estaciones.png",
]
faltantes_png_01 = [nombre for nombre in nombres_png_01
                    if not (SALIDAS_01 / nombre).is_file()]
assert not faltantes_png_01, f"PNG faltantes: {faltantes_png_01}"
assert len(tablas_exportar_01) == 10
print("CONTROL FINAL — Notebook 01")
print("Estaciones auditadas:", df_dim_estaciones_01["station_id"].nunique())
print("Registros interval históricos:", int(df_global_horizonte_01["n_interval"].sum()))
print("CSV: 10 | JSON: 1 | PNG: 8")
print("Carpeta de salida:", SALIDAS_01)
print("Test de noviembre: NO LEÍDO")
print("Datos del modelo final: NO MODIFICADOS")
print("Validar con Notebook 00 antes de reemplazar la versión anterior.")

## 10. Conclusión académica y vínculo con el dashboard

Este notebook mide de forma diferenciada **disponibilidad** y **precisión**. Las métricas principales se calculan sobre emparejamientos `interval` con temperaturas finitas; `fallback` queda como contraste explícito. La cobertura desigual, los posibles cambios de metadatos y la cantidad de días observados acompañan a los rankings por estación.

Los mapas y las comparaciones de horizontes constituyen **análisis descriptivos** del período histórico, no pruebas causales ni afirmaciones de generalización a estaciones nuevas. El análisis espacial aporta motivos para considerar latitud y longitud como predictores y para examinar heterogeneidad, pero no prueba su contribución incremental en el modelo; esa evidencia corresponde a los notebooks 04–08.

**Para presentar únicamente en dashboard:** utilizar las 10 tablas y el JSON exportados por esta etapa para construir mapa interactivo de estaciones con selector de horizonte, cobertura `interval`/`fallback`, filtros por estado y número de días, resumen de MAE/RMSE/sesgo, serie temporal con hueco 22–26/10, mapa de sesgos y comparaciones descriptivas. Los PNG son referencias visuales reproducibles, no sustitutos de la interactividad.

**Limitaciones:** las observaciones no son independientes, las estaciones no son una muestra espacial aleatoria, los umbrales para destacar estaciones son descriptivos, y la visualización geográfica depende de la calidad y estabilidad de los metadatos. Los ocho días de noviembre siguen completamente fuera de este notebook.

**Cierre condicionado:** aprobar y reemplazar la versión anterior solo después de ejecutar este archivo con los 61 CSV reales presentes (se leen los 48 históricos), cotejar las métricas con el Notebook 00 y comprobar las exportaciones. Las salidas antiguas no se reutilizan.